#Projeto Integrador de Ciência de Dados

#Informações da Dupla
202426610020 - Lucas Rivelo Campos Almeida

202426610052 - Thiago Cavalcante dos Santos




---



#Entendimento do Problema

##Problema Escolhido:
Desenvolver um modelo de machine learning que preve o estado clínico de cavalos com suspeita de cólica.

##Contexto do Problema

Usando de um dataset com 299 casos distintos, abordamos nesse trabalho, equinos que passaram por algum tipo de dor abdominal classificada como cólica, uma das principais causas de morte e atendimento médico veterinário. A base reúne prontuários médicos e categorias de grande importância para a determinação do desfecho do animal.

##Pergunta Principal
É possível prever o estado clínico [Lived, Euthanized ou Died] de um cavalo a partir das informações obtidas durante o atendimento presentes no dataset?

##Perguntas Secundárias
*  Qual modelo de classificação apresenta o melhor desempenho para esse dataset?
*  Como o fator humano interfere no resultado do algoritmo quando se tenta prever cavalos eutanaziados?

##Motivação da Escolha

Nosso objetivo está na busca por um modelo de machine learning que proporcione a melhor capacidade de previsão da sobrevivência de um cavalo, mesmo com o desafio de uma base com 30% dos seus valores faltantes.

##Relevância Prática e Acadêmica

Na prática, o desenvolvimento de um modelo com capacidade preditiva serviria como uma grande ferramenta aliada para os profissionais da área da medicina veterinária, auxiliando na identificação dos casos, otimizando processos de triagem e contribuindo na tomada de decisão.

##Tipo de tarefa: Classificação Multiclasse

##Variável-Alvo: "Outcome"
Coluna que determina as três classes: Lived, Euthanized e Died.

##Possíveis Usuários:


*   Hospitais e Clínicas Veterinárias;
*   Profissionais da área veterinária;
*   Pesquisadores em Medicina Veterinária e/ou Ciência de Dados;
*   Donos de Cavalos.


##Limitações Iniciais
O dataset apresenta 30% de valores ausentes, variando entre as colunas, dificultando a exploração dos dados que já não são demasiados de maneira geral (299 observações). Outro ponto é a determinação dos animais eutanasiados, já que parte de uma decisão humana.

##Resultados Esperados:
Esperamos identificar o modelo com melhor capacidade preditiva, comparando entre os diferentes modelos de classificação e compreender quais características clínicas possuem maior influência sobre a categorização do 'outcome' dos animais.



---



#Planejamento do Projeto


##Etapas Executadas:
Os dados foram coletados e carregados no trabalho através do dataset, fizemos uma análise exploratória inicial para compreender a base e posteriormente nos aprofundamos em distribuições das classes com pré-processamentos através de pipelines após selecionarmos as variáveis de maior correlação com o alvo para os mapas. Usamos também do tratamento de outliers e colunas com valores faltantes razoáveis, drop de outras colunas consideradas desnecessárias e testes completos de modelos de Machine Learning para obter o melhor resultado nos parâmetros estabelecidos e consequentemente na matriz de confusão.

Seguindo sempre as principais etapas do processo CRISP-DM:

*  entendimento do problema;
*  aquisição da base de dados;
*  análise exploratória dos dados;
*  preparação e limpeza da base;
*  treinamento dos modelos de classificação;
*  avaliação dos modelos;
*  interpretação dos resultados e elaboração das conclusões.


#Dados Necessários
Utilizaremos a base Horse Survival Dataset (UCI Horse Colic Dataset), contendo as informações clínicas de cavalos atendidos com suspeita de cólica.
Os dados foram disponibilizados em arquivos CSV e carregados no ambiente, sem a necessidade de integração de múltiplas bases de dados, nem a utilização de APIs ou coleta manual de informações.

https://www.kaggle.com/datasets/yasserh/horse-survival-dataset

Data de Acesso: 17/06/2026

#Ferramentas Utilizadas

* Pandas;
* NumPy;
* Matplotlib;
* Seaborn;
* Scikit-Learn.

#Métricas Utilizadas
* Accuracy;
* Precision;
* Recall;
* F1-score;
* Matriz de Confusão.

#Riscos e Dificuldades
* quantidade exorbitante de valores ausentes em algumas variáveis;
* pequena quantidade de registros (299 amostras);
* desbalanceamento entre as classes;
* necessidade de converter variáveis categóricas para formato numérico antes do treinamento dos modelos.

#Importação das Bibliotecas

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.naive_bayes import GaussianNB

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    classification_report
)

#Carregamento do dataset

In [ ]:
try:
    from google.colab import files
    print("Use a janela para enviar os arquivos da aula.")
    uploaded = files.upload()
except Exception:
    print("Se você estiver em Jupyter local, coloque os arquivos na mesma pasta do notebook.")

Use a janela para enviar os arquivos da aula.


In [ ]:
# Carregamento do arquivo CSV disponibilizado
caminho = "horse.csv"

df = pd.read_csv(caminho)

print("Dataset carregado com sucesso!")
print("Formato da base:", df.shape)

df.head()

#Explorações Iniciais

In [ ]:
df.info()

In [ ]:
print("Valores ausentes:")
display(df.isna().sum())

In [ ]:
df.describe()

#Distribuição das Classes
Realizamos uma distribuição das classes da variável determinante para ver o equilibrio e a disposição dos dados dentro do dataset. A resposta 'Lived' contempla a maior parte do dataset, deixando entendido que a maioria dos dados vieram de animais vivos.

In [ ]:
# Conta quantas vezes cada diagnóstico aparece na base
contagem_classes = df["outcome"].value_counts()

# Exibe o resultado da contagem
contagem_classes

In [ ]:
# Define o tamanho da figura do gráfico
plt.figure(figsize=(6, 6))

# Cria o gráfico de pizza com porcentagens e ângulo inicial de 90 graus
contagem_classes.plot(
    kind="pie",
    autopct=lambda pct: f"{pct:.1f}%",
    startangle=90,
    ylabel=""
)

# Adiciona o título e exibe o gráfico final
plt.title("Distribuição das classes")
plt.show()

#Separação da Variável-Alvo e das Variáveis de Entrada

In [ ]:
# Variável alvo
alvo = "outcome"

X = df.drop(columns=["outcome"]) #caracteristicas do objeto
y = df[alvo] #o que queremos acertar

print("Formato de X:", X.shape)
print("Formato de y:", y.shape)

#Mapa de Correlação de Variáveis
Aqui iremos ter uma visualização mais intuitiva da relação das colunas entre elas mesmas e pontos fortes no dataset, de maneira que quanto mais perto de 1, a relação se torna mais positiva e mais perto de -1, se torna negativa.

In [ ]:
# Criando uma cópia dos dados originais para correlação
dados_corr = X.copy()

# Adiciona a variável-alvo apenas para analisar correlação com o diagnóstico
dados_corr["target"] = y

# Calcula a correlação
corr = dados_corr.corr(numeric_only=True)

plt.figure(figsize=(14, 10))
sns.heatmap(
    corr,
    cmap="coolwarm",
    center=0,
    linewidths=0.3,
    cbar=True
)

plt.title("Mapa de correlação antes das variáveis derivadas")
plt.show()

#Tratamento Prévio de Variáveis
As três colunas a seguir possuem o maior número de valores ausentes do dataset, analisando a comparação com o alvo e como elas afetam o resultado, decidimos por manter "abdomo_appearance" e aplicar uma variável categórica "missing" para os valores ausentes e dropar as colunas "nasogastric_reflux_ph" e "abdomo_protein" com mais de 60% de valores faltantes.

In [ ]:
X["abdomo_appearance"] = X["abdomo_appearance"].fillna("missing")
X = X.drop(columns=["nasogastric_reflux_ph"])
X = X.drop(columns=["abdomo_protein"])

In [ ]:
tabela = pd.crosstab(
    X["abdomo_appearance"],
    y,
    dropna=False,
    normalize="index"
)

display(tabela)

#Correlação com o Alvo
Selecionamos abaixo as 5 variáveis de melhor correlação com o alvo para formarmos a visualização de maneira que a proteina total e o pulso se destacam.

In [ ]:
le = LabelEncoder()

y_num = le.fit_transform(y) #transforma variáveis nominais em numéricas

X = X.drop(columns=["hospital_number"])

dados_corr = X.copy()

dados_corr["target"] = y_num

corr = dados_corr.corr(numeric_only=True)

# Correlação das variáveis com o target
corr_target = (
    corr["target"]
    .drop("target")
    .sort_values(key=abs, ascending=False)
)

print("Top variáveis mais correlacionadas com o alvo:")
display(corr_target.head(10))


In [ ]:
#Seleciona as 5 variáveis mais correlacionadas com o target
top_variaveis = corr_target.abs().sort_values(ascending=False).head(5).index.tolist()

# Inclui o target para visualização
dados_top_corr = dados_corr[top_variaveis + ["target"]]

corr_top = dados_top_corr.corr(numeric_only=True)

plt.figure(figsize=(12, 9))
sns.heatmap(
    corr_top,
    cmap="coolwarm",
    center=0,
    annot=True,
    fmt=".2f",
    linewidths=0.5
)

plt.title("Mapa compacto: variáveis mais correlacionadas com o alvo")
plt.show()

#Separação de Treino e Teste
Mantendo o modelo aprendido em sala de aula, não alteramos os parâmetros do test_size (3 para 1) e o random_state.

In [ ]:
# Divide os dados: 75% para treinar o modelo e 25% para testar o desempenho
# O 'stratify' garante que a proporção de casos malignos seja igual nos dois grupos
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

# Exibe o tamanho dos conjuntos e a proporção das classes para conferência
print("Treino:", X_treino.shape)
print("Teste:", X_teste.shape)
print("Distribuição no treino:")
print(y_treino.value_counts(normalize=True))
print("\nDistribuição no teste:")
print(y_teste.value_counts(normalize=True))

#Tratamento de Outliers - Fator IQR
Usando do IQRCapper, fazemos um transformador simples como o ensinado em sala com as bases lógicas dos cálculos do IQR, primeiro e segundo quartis e limites inferior e superior. Logo em seguida, há uma função avaliadora auxiliar para os modelos de pipeline utilizados posteriormente.

In [ ]:
class IQRCapper(BaseEstimator, TransformerMixin):
    def __init__(self, fator=1.5):
        # Define a sensibilidade para detectar outliers (o padrão 1.5 é o clássico de Tukey)
        self.fator = fator

    def fit(self, X, y=None):
        # Converte para DataFrame para garantir que o cálculo de quartis funcione corretamente
        X = pd.DataFrame(X).copy()
        self.columns_ = X.columns

        # Calcula o primeiro (25%) e o terceiro (75%) quartil de cada coluna
        q1 = X.quantile(0.25)
        q3 = X.quantile(0.75)

        # Calcula o IQR (amplitude do intervalo onde estão 50% dos dados centrais)
        iqr = q3 - q1

        # Define as barreiras (limites) inferior e superior
        self.lower_ = q1 - self.fator * iqr
        self.upper_ = q3 + self.fator * iqr
        return self

    def transform(self, X):
        # Aplica os limites aprendidos no 'fit' aos dados atuais
        X = pd.DataFrame(X, columns=self.columns_).copy()

        # O método clip 'achata' os valores que estão fora das barreiras (cap)
        X = X.clip(lower=self.lower_, upper=self.upper_, axis=1)
        return X

In [ ]:

def avaliar_classificador(nome_modelo, modelo, X_teste, y_teste):

    y_pred = modelo.predict(X_teste)

    resultados = {
        "modelo": nome_modelo,
        "accuracy": accuracy_score(y_teste, y_pred),
        "precision": precision_score(y_teste, y_pred, average="weighted"),
        "recall": recall_score(y_teste, y_pred, average="weighted"),
        "f1": f1_score(y_teste, y_pred, average="weighted")
    }

    return resultados

#Pré-Processamento
Usamos dois modelos de pré-processamento quase identicos para padronização das colunas, permitindo que as pipelines já tenham o material correto para tratarem.

In [ ]:
#Separa as variáveis numéricas e categóricas
colunas_numericas = X.select_dtypes(include=["number"]).columns.tolist()
colunas_categoricas = X.select_dtypes(exclude=["number"]).columns.tolist()

# Define os passos de pré-processamento para diferentes tipos de colunas
preprocessamento = ColumnTransformer(
    transformers=[
        # Pipeline para colunas numéricas:
        # 1. 'imputer': preenche valores ausentes com a mediana
        ("num", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median"))
        ]), colunas_numericas),

        # 2. 'onehot': converte categorias em representação one-hot (binária)
        #    handle_unknown='ignore' evita erros para categorias não vistas no treino
        ("cat", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), colunas_categoricas)
    ]
)

In [ ]:
#Separa as variáveis numéricas e categóricas
colunas_numericas = X.select_dtypes(include=["number"]).columns.tolist()
colunas_categoricas = X.select_dtypes(exclude=["number"]).columns.tolist()

# Define os passos de pré-processamento para diferentes tipos de colunas
preprocessamento_SVC = ColumnTransformer(
    transformers=[
        # Pipeline para colunas numéricas:
        # 1. 'imputer': preenche valores ausentes com a mediana
        ("num", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median"))
        ]), colunas_numericas),

        # 2. 'onehot': converte categorias em representação one-hot (binária)
        #    drop="first": evita colunas desnecessárias (a primeira)
        #    handle_unknown='ignore' evita erros para categorias não vistas no treino
        ("cat", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(drop="first", handle_unknown="ignore"))
        ]), colunas_categoricas)
    ]
)

#Pipelines
Selecionamos 4 tipos de modelos de classificação para comparativo, em busca do que possui melhores atributos.

In [ ]:
# Pipeline da Decision Tree
modelo_tree = DecisionTreeClassifier(random_state=42, class_weight="balanced")

#Conseguimos simplificar o código com os pré-processamentos, permitindo uma melhor visualização.
pipeline_tree = Pipeline(steps=[
    ("preprocessamento", preprocessamento),
    ("modelo", modelo_tree)
])

pipeline_tree.fit(X_treino, y_treino)
y_pred_tree = pipeline_tree.predict(X_teste)

resultado_tree = avaliar_classificador("Decision Tree", pipeline_tree, X_teste, y_teste)
print(resultado_tree)

In [ ]:
# Pipeline da Random Forest
modelo_rf = RandomForestClassifier(random_state=42, class_weight="balanced")

pipeline_rf = Pipeline(steps=[
    ("preprocessamento", preprocessamento),
    ("modelo", modelo_rf)
])

pipeline_rf.fit(X_treino, y_treino)
y_pred_rf = pipeline_rf.predict(X_teste)

resultado_rf = avaliar_classificador("Random Forest", pipeline_rf, X_teste, y_teste)
print(resultado_rf)

In [ ]:
# Pipeline de SVC
modelo_svc = Pipeline(steps=[
    ("preprocessamento", preprocessamento_SVC),                # Padroniza os dados
    ("modelo", SVC(
        C=1.0,                                     # Parâmetro de regularização
        kernel="rbf",                              # Kernel rbf para melhor adaptação
        max_iter=10000,                            # Limite de iterações para o algoritmo convergir
        class_weight="balanced",                   # Ajusta pesos para o desequilíbrio das classes
        random_state=42                            # Garante reprodutibilidade
    ))
])

# Treina o pipeline do SVC usando os dados de treino
modelo_svc.fit(X_treino, y_treino)

# Avalia o desempenho usando a função de avaliação
resultado_svc = avaliar_classificador(
    "SVC rbf C=1.0",
    modelo_svc,
    X_teste,
    y_teste
)

# Exibe o resultado do terceiro modelo isolado
pd.DataFrame([resultado_svc])

In [ ]:
# Pipeline de Regressão 1.0
modelo_regride = Pipeline(steps=[
    ("preprocessamento", preprocessamento_SVC),                # Padroniza os dados
    ("modelo", LogisticRegression(
        C=1.0,                                     # Parâmetro de regularização
        max_iter=10000,                            # Limite de iterações para o algoritmo convergir
        solver="liblinear",                        # Algoritmo de otimização
        class_weight="balanced",                   # Ajusta pesos para o desequilíbrio das classes
        random_state=42                            # Garante reprodutibilidade
    ))
])

# Treina o pipeline de Regressão usando os dados de treino
modelo_regride.fit(X_treino, y_treino)

# Avalia o desempenho usando a função de avaliação
resultado_regride = avaliar_classificador(
    "Regressão Logística",
    modelo_regride,
    X_teste,
    y_teste
)

# Exibe o resultado do terceiro modelo isolado
pd.DataFrame([resultado_regride])

##Resultados obtidos
Após rodar os quatro modelos, podemos analisar seus atributos e comparar os pontos fortes e fracos de cada um.

###Modelo - Accuracy - Precision - Recall - F1
'Random Forest' - 0.6 - 0.56230 - 0.6 - 0.57350

'Decision Tree' - 0.54666 - 0.53989- 0.54666 - 0.54266

'Regressão Logística'	- 0.66666 - 0.68896	- 0.66666 - 0.67041 🌟

'SVC rbf C=1.0' - 0.37333	- 0.817778 - 0.37333 -	0.39825


De modo geral, a Regressão Logística foi o modelo que apresentou melhores parametros, mesmo com o 81% de Precisão do SVM, a acurácia do modelo de regressão se sobressai.

In [ ]:
# Conferência do Tratamento do pre-processamento.
transformador = modelo_regride.named_steps['preprocessamento']

# Vemos os dados como na memória
dados = transformador.transform(X_treino)

#Somamos e conferimos se há valores nulos pendentes
pd.DataFrame(dados).isna().sum().sum()

##Matriz de Confusão
Chegamos agora no momento de ver a capacidade de previsão do modelo escolhido (Regressão Logistica) em uma matriz de confusão.

In [ ]:
# Gera previsões para o conjunto de teste
y_pred_regride = modelo_regride.predict(X_teste)

# Cria a matriz de confusão comparando o real vs previsto
cm = confusion_matrix(y_teste, y_pred_regride)

classes = modelo_regride.classes_

# Configura a exibição visual da matriz
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=classes
)

# Plota o gráfico para análise de erros (Falsos Positivos e Negativos)
disp.plot(values_format="d")
plt.title("Matriz de Confusão — Regressão Logística inicial")
plt.show()

In [ ]:
print(classification_report(
    y_teste,
    y_pred_regride,
    target_names=["died", "euthanized", "lived"]
))

O modelo tem grande capacidade de prever sobre a sobrevivência dos cavalos, acertando 31 de 45, algo parecido com a previsão dos mortos (15 de 19). O ponto crítico acontece quando se tenta prever sobre os animais eutanaziados, como ainda é uma decisão humana que não só envolve fatores de saúde, mas também disposição, vontade, decisões pessoais e o montante que o proprietário pode pagar no tratamento, houve pouca precisão.

##Alteração dos Parâmetros
Aqui fazemos alterações no "C" em busca de um valor melhor.

In [ ]:
#Pré-Processamento SVC

# Definimos uma lista de dicionários com diferentes combinações de hiperparâmetros para testar
configuracoes_logistica = [
    {"nome": "Logística C=0.01", "C": 0.01},
    {"nome": "Logística C=0.1", "C": 0.1},
    {"nome": "Logística C=0.3", "C": 0.3},
    {"nome": "Logística C=0.5", "C": 0.5},
    {"nome": "Logística C=1.0", "C": 1.0},
    {"nome": "Logística C=2.0", "C": 2.0},
    {"nome": "Logística C=3.0", "C": 3.0},
    {"nome": "Logística C=5.0", "C": 5.0},
]

# Listas para armazenar os resultados das métricas e os objetos dos modelos treinados
resultados_logistica = []
modelos_logistica = {}

# O loop percorre cada configuração, cria o pipeline, treina e avalia
for cfg in configuracoes_logistica:
    modelo = Pipeline(steps=[
    ("preprocessamento", preprocessamento_SVC),
    ("modelo", LogisticRegression(
        C=cfg["C"],
        solver="liblinear",
        max_iter=10000,
        class_weight="balanced",
        random_state=42
    ))
])

# Treina o modelo com os dados de treino
    modelo.fit(X_treino, y_treino)

# Guarda o modelo no dicionário usando o nome como chave
    modelos_logistica[cfg["nome"]] = modelo

# Chama a função de avaliação e guarda os resultados na lista
    resultados_logistica.append(avaliar_classificador(cfg["nome"], modelo, X_teste, y_teste))

# Converte os resultados para um DataFrame e ordena pela acurácia para facilitar a comparação
pd.DataFrame(resultados_logistica).sort_values(by="accuracy", ascending=False)

In [ ]:
#Pré-Processamento Padrão

# Definimos uma lista de dicionários com diferentes combinações de hiperparâmetros para testar
configuracoes_logistica = [
    {"nome": "Logística C=0.01", "C": 0.01},
    {"nome": "Logística C=0.1", "C": 0.1},
    {"nome": "Logística C=0.3", "C": 0.3},
    {"nome": "Logística C=0.5", "C": 0.5},
    {"nome": "Logística C=1.0", "C": 1.0},
    {"nome": "Logística C=2.0", "C": 2.0},
    {"nome": "Logística C=3.0", "C": 3.0},
    {"nome": "Logística C=5.0", "C": 5.0},
]

# Listas para armazenar os resultados das métricas e os objetos dos modelos treinados
resultados_logistica = []
modelos_logistica = {}

# O loop percorre cada configuração, cria o pipeline, treina e avalia
for cfg in configuracoes_logistica:
    modelo = Pipeline(steps=[
    ("preprocessamento", preprocessamento),
    ("modelo", LogisticRegression(
        C=cfg["C"],
        solver="liblinear",
        max_iter=10000,
        class_weight="balanced",
        random_state=42
    ))
])

# Treina o modelo com os dados de treino
    modelo.fit(X_treino, y_treino)

# Guarda o modelo no dicionário usando o nome como chave
    modelos_logistica[cfg["nome"]] = modelo

# Chama a função de avaliação e guarda os resultados na lista
    resultados_logistica.append(avaliar_classificador(cfg["nome"], modelo, X_teste, y_teste))

# Converte os resultados para um DataFrame e ordena pela acurácia para facilitar a comparação
pd.DataFrame(resultados_logistica).sort_values(by="accuracy", ascending=False)

É observável que o parâmetro C=0.1 possui melhores valores do que os outros da lista, melhor até que o que usamos no modelo, mas que utiliza do pré processamento sem o drop=first, mostrando que o drop da primeira categoria custou accuracy.

##Testando Novamente

In [ ]:
# Pipeline de Regressão 0.1
modelo_regride2 = Pipeline(steps=[
    ("preprocessamento", preprocessamento),                # Padroniza os dados (Agora sem o modelo para SVC)
    ("modelo", LogisticRegression(
        C=0.1,                                     # Parâmetro de regularização
        max_iter=10000,                            # Limite de iterações para o algoritmo convergir
        solver="liblinear",                        # Algoritmo de otimização
        class_weight="balanced",                   # Ajusta pesos para o desequilíbrio das classes
        random_state=42                            # Garante reprodutibilidade
    ))
])

# Treina o pipeline de Regressão usando os dados de treino
modelo_regride2.fit(X_treino, y_treino)

# Avalia o desempenho usando a mesma função de avaliação
resultado_regride2 = avaliar_classificador(
    "Regressão Logística",
    modelo_regride2,
    X_teste,
    y_teste
)

# Exibe o resultado do terceiro modelo isolado
pd.DataFrame([resultado_regride2])

In [ ]:
# Gera previsões para o conjunto de teste
y_pred_regride2 = modelo_regride2.predict(X_teste)

# Cria a matriz de confusão comparando o real vs previsto
cm = confusion_matrix(y_teste, y_pred_regride2)

classes = modelo_regride2.classes_

# Configura a exibição visual da matriz
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=classes
)

# Plota o gráfico para análise de erros (Falsos Positivos e Negativos)
disp.plot(values_format="d")
plt.title("Matriz de Confusão — Regressão Logística inicial")
plt.show()

##Como esperado, esse foi o melhor resultado que conseguimos chegar.
As poucas alterações que foram realizadas possibilitaram uma melhor performance dos atributos, variáveis como abdomo_appearance, pulse, packed_cell_volume e total_protein apresentaram relação importante com o desfecho clínico.
As limitações permanecem as mesmas, 30% dos dados faltantes somado a uma base não tão extensa trouxe deficiências de contexto e previsão para o projeto. Em atividades futuras, seria interessante avaliar modelos mais complexos e completos com bases mais robustas e usando de técnicas mais avançadas em busca de melhor aproveitamento.

Outras alternativas como a criação de novas variáveis, agrupamento de outras de menor incidencia ou do teste de mais modelos (KNN, MLP, Gradient Boosting, Naive Bayes - variações de Random Forest, Decision Tree e SVM)  foram realizadas, mas não foi obtido nenhum valor expressivo ou que superasse o nosso resultado atual.
Privamos por manter o código sem eles para melhor visualização de maneira limpa e para evitar os loops infinitos que estavam acontecendo.

Apesar dos entraves encontrados, ainda sim conseguimos fazer um trabalho viável com conhecimentos científicos da base de dados e do conteúdo estudado, atingindo uma capacidade preditiva de praticamente 70%, a contraparte dos dados faltantes.